# Proyecto Final: Transformación y Modelado de Indicadores Socioeconómicos en Latinoamérica

**Curso:** CC3074 - Minería de Datos  
**Framework:** CRISP-DM - Fase 3: Preparación de Datos  
**Fuente de datos:** CEPALSTAT - Observatorio de Desarrollo Digital (ODD)  
**Indicador:** Personas usuarias de Internet por grupo etario, países seleccionados de América Latina y el Caribe  

**Integrantes:**

- Cristian Túnchez (231359)  
- Dulce Ambrosio (231143)  
- Daniel Chet (231177)  
- Javier Linares (231135)  

## Semana 2 - Transformación del Conjunto de Datos: Construcción del Conjunto Analítico

---
### Contexto y Objetivo

En la Semana 1 se realizó el Análisis Exploratorio de Datos (EDA) sobre los **870 registros × 8 columnas** publicados por CEPAL. Se identificaron los siguientes hallazgos críticos para la fase de preparación de datos:

- **Cuatro columnas no informativas:** `indicator`, `unit`, `source_id` (constantes) y `notes_ids` (100% nula).
- **Panel desbalanceado:** sólo el **49.3%** de las combinaciones país × año × grupo etario están reportadas (870 de 1,764 posibles; 145 de 294 pares país-año).
- **Formato largo (long/tidy):** una única columna numérica `value` con dimensión categórica `Grupos etarios Uso Internet`, lo que dificulta el análisis multivariado.
- **Tipos de datos heterogéneos:** nombres de columnas con caracteres especiales y dobles guiones bajos.

**Objetivo de la Semana 2:** ejecutar la **Fase 3 completa de CRISP-DM (Preparación de Datos)**, dejando el conjunto de datos listo para la Fase de Modelado en la Semana 3.

**Configuración del entorno e importación de librerías**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator
from scipy import stats as sp_stats
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# Configuracion visual
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Paleta personalizada para grupos etarios (de joven a mayor)
PALETA_EDAD = {
    'edad de medicion a 17 años': '#2196F3',
    '18 a 25 años de edad': '#4CAF50',
    '26 a 50 años de edad': '#FF9800',
    '51 a 65 años': '#F44336',
    '66 años en adelante': '#9C27B0',
    'Total': '#607D8B'
}

ORDEN_EDAD = [
    'edad de medicion a 17 años',
    '18 a 25 años de edad',
    '26 a 50 años de edad',
    '51 a 65 años',
    '66 años en adelante',
    'Total'
]

# Etiquetas cortas para gráficos
ETIQUETAS_CORTAS = {
    'edad de medicion a 17 años': '<=17 años',
    '18 a 25 años de edad': '18-25',
    '26 a 50 años de edad': '26-50',
    '51 a 65 años': '51-65',
    '66 años en adelante': '66+',
    'Total': 'Total'
}

print('Entorno configurado correctamente.')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')

Entorno configurado correctamente.
pandas: 3.0.0
numpy: 2.4.2


**Carga de datos desde el archivo Excel (hoja `datos`)**

In [ ]:
df = pd.read_excel('data.xlsx', sheet_name='datos')

print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nColumnas originales:')
for c in df.columns:
    print(f'  - {c}')
print(f'\nPrimeras filas:')
display(df.head())

Dataset cargado: 870 filas x 8 columnas

Columnas originales:
  - indicator
  - País__ESTANDAR
  - Grupos etarios Uso Internet
  - Años__ESTANDAR
  - value
  - unit
  - notes_ids
  - source_id

Primeras filas:


,indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
0,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2016,75.9,Porcentaje sobre el total de personas en cada ...,NaN,9353
1,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2017,75.9,Porcentaje sobre el total de personas en cada ...,NaN,9353
2,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2018,78.5,Porcentaje sobre el total de personas en cada ...,NaN,9353
3,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2019,78.9,Porcentaje sobre el total de personas en cada ...,NaN,9353
4,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2020,87.5,Porcentaje sobre el total de personas en cada ...,NaN,9353


**Recapitulación del estado inicial**

In [ ]:
print('=' * 70)
print('ESTADO INICIAL DEL DATASET')
print('=' * 70)
print(f'\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\nValores únicos por columna:')
print('─' * 50)
for col in df.columns:
    n_unique = df[col].nunique()
    n_null = df[col].isna().sum()
    print(f'  {col:35s}  únicos: {n_unique:4d}  nulos: {n_null:4d}')

print(f'\nMemoria utilizada: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')

ESTADO INICIAL DEL DATASET

Dimensiones: 870 filas x 8 columnas

Valores únicos por columna:
──────────────────────────────────────────────────
  indicator                            únicos:    1  nulos:    0
  País__ESTANDAR                       únicos:   14  nulos:    0
  Grupos etarios Uso Internet          únicos:    6  nulos:    0
  Años__ESTANDAR                       únicos:   21  nulos:    0
  value                                únicos:  577  nulos:    0
  unit                                 únicos:    1  nulos:    0
  notes_ids                            únicos:    0  nulos:  870
  source_id                            únicos:    1  nulos:    0

Memoria utilizada: 495.7 KB
